# Dependencies

In [ ]:
import requests
import csv
from tqdm import tqdm 
from pykml import parser
import folium
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon
import geopy.distance

# Fetch MTR stations data from API

In [2]:
# MTR stations grouping by districts
districts_to_stations = {
    "Kwai Tsing": [
        "Kwai Hing", "Kwai Fong", "Lai King", "Tsing Yi"
    ], 
    "North": [
        "Lo Wu", "Sheung Shui", "Fanling"
    ], 
    "Sai Kung": [
        "Tiu Keng Leng", "LOHAS Park", "Po Lam", "Hang Hau", "Tseung Kwan O"
    ],
    "Sha Tin": [
        'University', 'Racecourse', 'Fo Tan', 'Sha Tin', 'Tai Wai', 'Hin Keng', 'Che Kung Temple', 'Sha Tin Wai', 'City One', 'Shek Mun', 'Tai Shui Hang', 'Heng On', 'Ma On Shan', 'Wu Kai Sha'
    ],
    'Tai Po': [
        'Tai Wo', 'Tai Po Market'
    ],
    'Tsuen Wan': [
        'Tsuen Wan', 'Tai Wo Hau', 'Sunny Bay', 'Tsuen Wan West', 'Disneyland Resort'
    ],
    'Tuen Mun': [
        'Tuen Mun', 'Siu Hong'
    ],
    'Yuen Long': [
        'Lok Ma Chau', 'Tin Shui Wai', 'Long Ping', 'Yuen Long', 'Kam Sheung Road'
    ],
    'Kowloon City': [
        'Kowloon Tong', 'Ho Man Tin', 'Whampoa', 'To Kwa Wan', 'Sung Wong Toi', 'Kai Tak'
    ],
    'Kwun Tong': [
        'Yau Tong', 'Lam Tin', 'Kwun Tong', 'Ngau Tau Kok', 'Kowloon Bay', 'Choi Hung', 'Yau Tong'
    ],
    'Sham Shui Po': [
        'Kowloon Tong', 'Shek Kip Mei', 'Mei Foo', 'Lai Chi Kok', 'Cheung Sha Wan', 'Sham Shui Po', 'Nam Cheong'
    ],
    'Wong Tai Sin': [
        'Choi Hung', 'Diamond Hill', 'Wong Tai Sin', 'Lok Fu'
    ],
    'Yau Tsim Mong': [
        'Mong Kok East', 'Hung Hom', 'Prince Edward', 'Mong Kok', 'Yau Ma Tei', 'Jordan', 'Tsim Sha Tsui', 'Olympic', 'Kowloon', 'Austin', 'East Tsim Sha Tsui'
    ],
    'Central and Western': [
        'Admiralty', 'Central', 'Sheung Wan', 'Sai Ying Pun', 'HKU', 'Kennedy Town', 'Hong Kong'
    ],
    'Eastern': [
        'Chai Wan', 'Heng Fa Chuen', 'Shau Kei Wan', 'Sai Wan Ho', 'Tai Koo', 'Quarry Bay', 'North Point', 'Fortress Hill'
    ],
    'Southern': [
        'South Horizons', 'Lei Tung', 'Wong Chuk Hang', 'Ocean Park'
    ],
    'Wan Chai': [
        'Exhibition Centre', 'Tin Hau', 'Causeway Bay', 'Wan Chai'
    ],
    'Islands': [
        'Tung Chung', 'AsiaWorld-Expo', 'Airport'
    ]
}

# map station names to their corresponding districts
station_to_district = {station: district for district, stations in districts_to_stations.items() for station in stations}
stations = []
unique_stations_value = set()
full_stations = []
unique_stations_name = set()
    
def get_station_name_by_id(station_id):
    for station in stations:
        if str(station['value']) == str(station_id):  
            return station['label']
    return None

In [ ]:
# Load stations data from mtr_lines_and_stations.csv, collected from DATA.GOV.HK 
with open('MTR_Data/mtr_lines_and_stations.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        value = row["Station ID"]
        if value not in unique_stations_value and value != '': 
            unique_stations_value.add(value)  
            station = {
                "label": row["English Name"],
                "type": "HRStation",
                "line": row["Station Code"],
                "value": row["Station ID"]
            }
            stations.append(station)
            
            if station['label'] not in unique_stations_name:
                unique_stations_name.add(station['label'])
                full_stations.append(station)

# Load MTR routes data from the API            
with open('MTR_Data/mtr_station_info.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)

    writer.writerow([
        "Origin Station",
        "Origin District",
        "Destination Station",
        "Destination District",
        "Route Travel Time (mins)",
        "Fare (HKD)",
        "Full Route" 
    ])

    for origin in tqdm(full_stations, desc="Loading Origin Stations", unit="station"):
        for destination in tqdm(full_stations, desc="Loading Destination Stations", unit="station", leave=False):       
            if origin == destination:
                continue
            
            url = f"https://www.mtr.com.hk/share/customer/jp/api/HRRoutes/?o={origin['value']}&d={destination['value']}&lang=E"

            response = requests.get(url)

            if response.status_code == 200:
                data = response.json()
                        
                origin_name = origin['label']
                origin_district = station_to_district[origin_name]
                destination_name = destination['label']
                destination_district = station_to_district[destination_name]
                
                routes = data.get('routes', []) 

                if routes:
                    first_route = routes[0] # get fastest route 
                    time = first_route.get('time')  
                    #fare = first_route.get('fares', [])[0]["fareInfo"]["adult"]["octopus"]
                    
                    fare_info = first_route.get('fares', [])
                    if len(fare_info) != 0:
                        fare = fare_info[0]["fareInfo"]["adult"]["octopus"]
                    else:
                        fare = 0 
                    
                    path = first_route.get('path', [])  
                    full_route = []
                    
                    for stop in path:
                        station_name = get_station_name_by_id(stop.get('ID'))
                        if station_name:
                            full_route.append(station_name)
                            
                    full_route2 = ' -> '.join(full_route)

                    writer.writerow([
                        origin_name, origin_district, 
                        destination_name, destination_district,
                        time, fare,
                        full_route2 
                    ])
        
            else:
                print(f"Error: {response.status_code}")

# Get coordinates of MTR stations, and map corresponding constituency

In [ ]:
with open("MTR_Data/DCGC_2023.kml") as f:
    kml = parser.parse(f)
    districts = {}
    for p in kml.getroot().Document.Folder.Placemark:
        data = {d.values()[0] : d.text for d in p.ExtendedData.SchemaData.SimpleData}
        districts[data['DCGCCODE']] = {
            **data,
            'line': np.array(list(
                        map(lambda x: [float(y) for y in x.split(',')[:2]][::-1],
                        p.Polygon.outerBoundaryIs.LinearRing.coordinates.text.split(' ')))),
            'polygon': Polygon(
                        map(lambda x: [float(y) for y in x.split(',')[:2]],
                        p.Polygon.outerBoundaryIs.LinearRing.coordinates.text.split(' ')))
        }
        
def district_of_point(x, y):
    try:
        return next(filter(lambda d: d['polygon'].contains(Point(x, y)), districts.values()))['ENAME']
    except StopIteration:
        return "Shenzhen"
    
stations = []
unique_stations = set()

with open('MTR_Data/mtr_lines_and_stations.csv', mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        name = row["English Name"]
        if name not in unique_stations:
            unique_stations.add(name)
            station = {
                "label": row["English Name"],
                "value": row["Station ID"]
            }
            stations.append(station)

with open('MTR_Data/mtr_station_coords.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow([
        "Station Name",
        "Station Number",
        "Latitude",
        "Longitude",
        "Constituency"
    ])
    
    origin = stations[0]
    origin_coords = [22.321713,113.942717]
    origin_district = district_of_point(origin_coords[1], origin_coords[0])
    writer.writerow([
        origin["label"],
        origin["value"],
        origin_coords[0],
        origin_coords[1],
        origin_district
    ])
    
    for destination in tqdm(stations[1:], desc="Loading Origin Stations", unit="station"):

        url = f"https://www.mtr.com.hk/share/customer/jp/api/CompleteRoutes/?lang=E&oLabel={origin['label']}&oType=HRStation&oValue={origin['value']}&dLabel={destination['label']}&dType=HRStation&dValue={destination['value']}"
        
        response = requests.get(url)

        if response.status_code == 200:
            data = response.json()

            destination_name = destination['label']
            destination_value = destination['value']
            destination_coords = data.get("dCoordinate", "").split(",")
            destination_district = district_of_point(float(destination_coords[1]), float(destination_coords[0]))

            if destination_coords and len(destination_coords) == 2:
                writer.writerow([
                    destination_name,
                    destination_value,
                    destination_coords[0],
                    destination_coords[1], 
                    destination_district
                ])
        else:
            print(f"Error: {response.status_code}")


# Get interstations distances using Citymapper API

In [ ]:
# calculate the distance between two points along the path
def calculate_distance_between_stations(path, start_index, end_index):
    distance = 0
    for i in range(start_index, end_index):
        distance += geopy.distance.geodesic(path[i], path[i + 1]).km
    return distance

# get the index of a station in the path
def get_station_index(stop_points, station_id):
    for point in stop_points:
        if point['id'] == station_id:
            return point['path_index']
    return -1

# URLs and station mappings for each MTR line (except East Rail Line & Tseung Kwan O Line)
mtr_lines = [
    # Airport Express Line
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E6%A9%9F%E5%A0%B4%E5%BF%AB%E7%B6%AB-airport-express&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1",
        "station_names": {
            'HKStation_BoLanGuanAsiaworldexpo': 'AsiaWorld-Expo',
            'HKStation_JiChangAirport': 'Airport',
            'HKStation_QingYiTsingYi': 'Tsing Yi',
            'HKStation_JiuLongKowloon': 'Kowloon', 
            'HKStation_XiangGangHongKong': 'Hong Kong'
        }
    },
    # Disneyland Resort Line
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E8%BF%AA%E5%A3%AB%E5%B0%BC%E7%B6%AB-disneyland-resort-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1",
        "station_names": {
            'HKStation_XinAoSunnyBay': 'Sunny Bay',
            'CMStation_di_shi_ni_disneyland_resort_1': 'Disneyland Resort'
        }
    },
    # Island Line
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E6%B8%AF%E5%B3%B6%E7%B6%AB-island-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1",
        "station_names": {
            'CMStation_jian_ni_di_cheng_kennedy_town': 'Kennedy Town',
            'CMStation_xiang_gang_da_xue_hku': 'HKU',
            'CMStation_xi_ying_pan_sai_ying_pun': 'Sai Ying Pun', 
            'HKStation_ShangHuanSheungWan': 'Sheung Wan',
            'HKStation_ZhongHuanCentral': 'Central', 
            'HKStation_JinZhongAdmiralty': 'Admiralty',
            'HKStation_WanZiWanChai': 'Wan Chai',
            'HKStation_TongLuoWanCausewayBay': 'Causeway Bay', 
            'HKStation_TianHouTinHau': 'Tin Hau',
            'HKStation_PaoTaiShanFortressHill': 'Fortress Hill',
            'HKStation_BeiJiaoNorthPoint': 'North Point', 
            'HKStation_ZeYuYongQuarryBay': 'Quarry Bay', 
            'HKStation_TaiGuTaiKoo': 'Tai Koo',
            'HKStation_XiWanHeSaiWanHo': 'Sai Wan Ho',
            'HKStation_ShaoJiWanShauKeiWan': 'Shau Kei Wan',
            'HKStation_XingHuaCunHengFaChuen': 'Heng Fa Chuen',
            'HKStation_ChaiWanChaiWan': 'Chai Wan'
        }
    }, 
    # Kwun Tong Line
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E8%A7%80%E5%A1%98%E7%B6%AB-kwun-tong-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1", 
        "station_names": {
            'CMStation_huang_bu_whampoa': 'Whampoa',
            'CMStation_he_wen_tian_ho_man_tin': 'Ho Man Tin',
            'HKStation_YouMaDiYauMaTei': 'Yau Ma Tei',
            'HKStation_WangJiaoMongKok': 'Mong Kok',
            'HKStation_TaiZiPrinceEdward': 'Prince Edward',
            'HKStation_ShiXiaWeiShekKipMei': 'Shek Kip Mei',
            'HKStation_JiuLongTangKowloonTong': 'Kowloon Tong',
            'HKStation_LeFuLokFu': 'Lok Fu',
            'HKStation_HuangDaXianWongTaiSin': 'Wong Tai Sin',
            'HKStation_ZuanShiShanDiamondHill': 'Diamond Hill',
            'HKStation_CaiHongChoiHung': 'Choi Hung',
            'HKStation_JiuLongWanKowloonBay': 'Kowloon Bay',
            'HKStation_NiuTouJiaoNgauTauKok': 'Ngau Tau Kok',
            'HKStation_GuanTangKwunTong': 'Kwun Tong',
            'HKStation_LanTianLamTin': 'Lam Tin',
            'HKStation_YouTangYauTong': 'Yau Tong',
            'HKStation_DiaoJingLingTiuKengLeng': 'Tiu Keng Leng'
        }
    }, 
    # South Island Line 
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E5%8D%97%E6%B8%AF%E5%B3%B6%E7%B6%AB-south-island-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1", 
        "station_names": {
             'HKStation_JinZhongAdmiralty': 'Admiralty',
            'CMStation_hai_yang_gong_yuan_ocean_park': 'Ocean Park',
            'CMStation_huang_zhu_keng_wong_chuk_hang': 'Wong Chuk Hang',
            'CMStation_li_dong_lei_tung': 'Lei Tung', 
            'CMStation_hai_yi_ban_dao_south_horizons': 'South Horizons'
        }
    }, 
    # Tuen Mun Line 
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E5%B1%AF%E9%A6%AC%E7%B6%AB-tuen-ma-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1", 
        "station_names": {
            'HKStation_WuXiShaWuKaiSha': 'Wu Kai Sha',
            'HKStation_MaAnShanMaOnShan': 'Ma On Shan',
            'HKStation_HengAnHengOn': 'Heng On',
            'HKStation_DaShuiKengTaiShuiHang': 'Tai Shui Hang',
            'HKStation_ShiMenShekMun': 'Shek Mun',
            'HKStation_DiYiChengCityOne': 'City One',
            'HKStation_ShaTianWeiShaTinWai': 'Sha Tin Wai',
            'HKStation_CheGongMiaoCheKungTemple': 'Che Kung Temple',
            'HKStation_DaWeiTaiWai': 'Tai Wai',
            'CMStation_xian_jing_hin_keng': 'Hin Keng',
            'HKStation_ZuanShiShanDiamondHill': 'Diamond Hill',
            'CMStation_qi_de_kai_tak': 'Kai Tak',
            'CMStation_song_huang_tai_sung_wong_toi': 'Sung Wong Toi',
            'CMStation_tu_gua_wan_to_kwa_wan': 'To Kwa Wan',
            'CMStation_he_wen_tian_ho_man_tin': 'Ho Man Tin',
            'HKStation_HongKanHungHom': 'Hung Hom',
            'HKStation_JianDongEastTsimShaTsui': 'East Tsim Sha Tsui',
            'HKStation_KeShiDianAustin': 'Austin',
            'HKStation_NanChangNamCheong': 'Nam Cheong',
            'HKStation_MeiFuMeiFoo': 'Mei Foo',
            'HKStation_QuanWanXiTsuenWanWest': 'Tsuen Wan West',
            'HKStation_JinShangLuKamSheungRoad': 'Kam Sheung Road',
            'HKStation_YuanLangYuenLong': 'Yuen Long',
            'HKStation_LangPingLongPing': 'Long Ping',
            'HKStation_TianShuiWeiTinShuiWai': 'Tin Shui Wai',
            'HKStation_ZhaoKangSiuHong': 'Siu Hong',
            'HKStation_TunMenTuenMun': 'Tuen Mun'
        }
    }, 
    # Tung Chung Line 
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E6%9D%B1%E6%B6%8C%E7%B6%AB-tung-chung-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1", 
        "station_names": {
            'HKStation_XiangGangHongKong': 'Hong Kong',
            'HKStation_JiuLongKowloon': 'Kowloon',
            'HKStation_AoYunOlympic': 'Olympic',
            'HKStation_NanChangNamCheong': 'Nam Cheong',
            'HKStation_LiJingLaiKing': 'Lai King',
            'HKStation_QingYiTsingYi': 'Tsing Yi',
            'HKStation_XinAoSunnyBay': 'Sunny Bay',
            'HKStation_DongYongTungChung': 'Tung Chung'
        }
    }, 
    # Tsuen Wan Line 
    {
        "url": "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E8%8D%83%E7%81%A3%E7%B6%AB-tsuen-wan-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1", 
        "station_names": {
            'HKStation_ZhongHuanCentral': 'Central',
            'HKStation_JinZhongAdmiralty': 'Admiralty',
            'HKStation_JianShaJuTsimShaTsui': 'Tsim Sha Tsui',
            'HKStation_ZuoDunJordan': 'Jordan',
            'HKStation_YouMaDiYauMaTei': 'Yau Ma Tei',
            'HKStation_WangJiaoMongKok': 'Mong Kok',
            'HKStation_TaiZiPrinceEdward': 'Prince Edward',
            'HKStation_ShenShuiBuShamShuiPo': 'Sham Shui Po',
            'HKStation_ChangShaWanCheungShaWan': 'Cheung Sha Wan',
            'HKStation_LiZhiJiaoLaiChiKok': 'Lai Chi Kok',
            'HKStation_MeiFuMeiFoo': 'Mei Foo',
            'HKStation_LiJingLaiKing': 'Lai King',
            'HKStation_KuiFangKwaiFong': 'Kwai Fong',
            'HKStation_KuiXingKwaiHing': 'Kwai Hing',
            'HKStation_DaWoKouTaiWoHau': 'Tai Wo Hau',
            'HKStation_QuanWanTsuenWan': 'Tsuen Wan'
        }
    }
]

# Open a single CSV file for all MTR lines
with open('MTR_Data/mtr_station_distances.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow([
        "Origin Station",
        "Origin X_coord",
        "Origin Y_coord",
        "Destination Station",
        "Destination X_coord",
        "Destination Y_coord",
        "Distance (km)"
    ])
    
    # Loop through each MTR line
    for line in mtr_lines:
        response = requests.get(line["url"])
        if response.status_code == 200:
            data = response.json()
            
            path = data["routes"][0]["patterns"][0]["path"]
            stop_points = data["routes"][0]["patterns"][0]["stop_points"]
            station_names = line["station_names"]
            station_ids = list(station_names.keys())
            
            for id in range(len(station_ids) - 1):
                start_station_id = station_ids[id]
                end_station_id = station_ids[id + 1]
                
                start_index = get_station_index(stop_points, start_station_id)
                end_index = get_station_index(stop_points, end_station_id)
                
                distance = calculate_distance_between_stations(path, start_index, end_index)
                origin_station = station_names[start_station_id]
                origin_x_coord = path[start_index][0]
                origin_y_coord = path[start_index][1]
                destination_station = station_names[end_station_id]
                destination_x_coord = path[end_index][0]
                destination_y_coord = path[end_index][1]
                
                writer.writerow([
                    origin_station,
                    origin_x_coord,
                    origin_y_coord,
                    destination_station, 
                    destination_x_coord,
                    destination_y_coord,
                    distance
                ])
                
                writer.writerow([
                    destination_station,
                    destination_x_coord,
                    destination_y_coord,
                    origin_station, 
                    origin_x_coord,
                    origin_y_coord,
                    distance
                ])
        else:
            print(f"Error fetching data for URL: {line['url']} - Status Code: {response.status_code}")

In [ ]:
# process a single route pattern
def process_route_pattern(writer, path, stop_points, station_names):
    station_ids = list(station_names.keys())
    total_dist = 0

    for id in range(len(station_ids) - 1):
        start_station_id = station_ids[id]
        end_station_id = station_ids[id + 1]

        start_index = get_station_index(stop_points, start_station_id)
        end_index = get_station_index(stop_points, end_station_id)

        distance = calculate_distance_between_stations(path, start_index, end_index)
        origin_station = station_names[start_station_id]
        origin_x_coord = path[start_index][0]
        origin_y_coord = path[start_index][1]
        destination_station = station_names[end_station_id]
        destination_x_coord = path[end_index][0]
        destination_y_coord = path[end_index][1]

        writer.writerow([
            origin_station,
            origin_x_coord,
            origin_y_coord,
            destination_station,
            destination_x_coord,
            destination_y_coord,
            distance
        ])

        writer.writerow([
            destination_station,
            destination_x_coord,
            destination_y_coord,
            origin_station,
            origin_x_coord,
            origin_y_coord,
            distance
        ])

        total_dist += distance

    return total_dist

# Get inter station distances for East Rail Line
url = "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E6%9D%B1%E9%90%B5%E7%B6%AB-east-rail-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1"
response = requests.get(url)

with open('MTR_Data/mtr_station_distances.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)

    writer.writerow([
        "Origin Station",
        "Origin X_coord",
        "Origin Y_coord",
        "Destination Station",
        "Destination X_coord",
        "Destination Y_coord",
        "Distance (km)"
    ])

    if response.status_code == 200:
        data = response.json()

        # Define the patterns and station names
        patterns = [
            {
                "path": data["routes"][0]["patterns"][2]["path"],
                "stop_points": data["routes"][0]["patterns"][2]["stop_points"],
                "station_names": {
                    'HKStation_JinZhongAdmiralty': 'Admiralty',
                    'CMStation_hui_zhan_exhibition_centre': 'Exhibition Centre',
                    'HKStation_HongKanHungHom': 'Hung Hom',
                    'HKStation_WangJiaoDongMongKokEast': 'Mong Kok East',
                    'HKStation_JiuLongTangKowloonTong': 'Kowloon Tong',
                    'HKStation_DaWeiTaiWai': 'Tai Wai',
                    'HKStation_ShaTianShaTin': 'Sha Tin',
                    'HKStation_HuoTanFoTan': 'Fo Tan',
                    'HKStation_DaXueUniversity': 'University',
                    'HKStation_DaBuXuTaiPoMarket': 'Tai Po Market',
                    'HKStation_TaiHeTaiWo': 'Tai Wo',
                    'HKStation_FenLingFanling': 'Fanling',
                    'HKStation_ShangShuiSheungShui': 'Sheung Shui',
                    'HKStation_LuoMaZhouLokMaChau': 'Lok Ma Chau'
                }
            },
            {
                "path": data["routes"][0]["patterns"][1]["path"],
                "stop_points": data["routes"][0]["patterns"][1]["stop_points"],
                "station_names": {
                    'HKStation_ShangShuiSheungShui': 'Sheung Shui',
                    'HKStation_LuoHuLoWu': 'Lo Wu'
                }
            },
            {
                "path": data["routes"][0]["patterns"][0]["path"],
                "stop_points": data["routes"][0]["patterns"][0]["stop_points"],
                "station_names": {
                    'HKStation_ShaTianShaTin': 'Sha Tin',
                    'CMStation_ma_chang_racecourse': 'Racecourse',
                    'HKStation_DaXueUniversity': 'University'
                }
            }
        ]

        total_distance = 0
        for pattern in patterns:
            total_distance += process_route_pattern(
                writer,
                pattern["path"],
                pattern["stop_points"],
                pattern["station_names"]
            )

    else:
        print(f"Error: {response.status_code}")

In [ ]:
# Get inter station distances for Tseung Kwan O Line
url = "https://citymapper.com/api/1/routeinfo?route=%E6%B8%AF%E9%90%B5-mtr-%E5%B0%87%E8%BB%8D%E6%BE%B3%E7%B6%AB-tseung-kwan-o-line&region_id=hk-hongkong&weekend=1&status_format=rich&extended=1"
response = requests.get(url)

with open('MTR_Data/mtr_station_distances.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)

    writer.writerow([
        "Origin Station",
        "Origin X_coord",
        "Origin Y_coord",
        "Destination Station",
        "Destination X_coord",
        "Destination Y_coord",
        "Distance (km)"
    ])

    if response.status_code == 200:
        data = response.json()

        # Define the patterns and station names
        patterns = [
            {
                "path": data["routes"][0]["patterns"][0]["path"]
                "stop_points": data["routes"][0]["patterns"][0]["stop_points"]
                "station_names": {
                    'HKStation_BeiJiaoNorthPoint': 'North Point',
                    'HKStation_ZeYuYongQuarryBay': 'Quarry Bay',
                    'HKStation_YouTangYauTong': 'Yau Tong',
                    'HKStation_DiaoJingLingTiuKengLeng': 'Tiu Keng Leng',
                    'HKStation_JiangJunAoTseungKwanO': 'Tseung Kwan O',
                    'HKStation_KengKouHangHau': 'Hang Hau',
                    'HKStation_BaoLinPoLam': 'Po Lam'
                    #'HKStation_KangChengLohasPark': 'LOHAS Park'
                }
            },
            {
                "path": data["routes"][0]["patterns"][2]["path"]
                "stop_points": data["routes"][0]["patterns"][2]["stop_points"]
                "station_names": {
                    'HKStation_KangChengLohasPark': 'LOHAS Park',
                    'HKStation_JiangJunAoTseungKwanO': 'Tseung Kwan O',
                }
            }
        ]

        total_distance = 0
        for pattern in patterns:
            total_distance += process_route_pattern(
                writer,
                pattern["path"],
                pattern["stop_points"],
                pattern["station_names"]
            )

    else:
        print(f"Error: {response.status_code}")

# Combine collected data into MTR_Data csv

In [ ]:
mtr_data = pd.read_csv("MTR_Data/mtr_station_info.csv")
distances = pd.read_csv("MTR_Data/mtr_station_distances.csv")
station_coords = pd.read_csv("MTR_Data/mtr_station_coords.csv")

distance_dict = {}
for _, row in distances.iterrows():
    origin = row["Origin Station"]
    destination = row["Destination Station"]
    distance = row["Distance (km)"]
    distance_dict[(origin, destination)] = distance

# Function to calculate total distances and stations
def calculate_total_distances_and_stations(route):
    stations = route.split(" -> ")
    total_distance = 0
    for i in range(len(stations) - 1):
        origin = stations[i]
        destination = stations[i + 1]
        total_distance += distance_dict.get((origin, destination), 0) 
    return total_distance, len(stations)

mtr_data["Total Distance (km)"], mtr_data["No. of Stations"] = zip(*mtr_data["Full Route"].apply(calculate_total_distances_and_stations))

mtr_data.rename(columns={"Route Travel Time (mins)": "Total Travel Time (mins)"}, inplace=True)

# Merge with station coordinates
mtr_data = mtr_data.merge(station_coords, left_on="Origin Station", right_on="Station Name", how="left").rename(columns={
    "Station Number": "Origin Station Number",
    "Latitude": "Origin Latitude",
    "Longitude": "Origin Longitude",
    "Constituency": "Origin Constituency"
}).drop(columns=["Station Name"])

mtr_data = mtr_data.merge(station_coords, left_on="Destination Station", right_on="Station Name", how="left").rename(columns={
    "Station Number": "Destination Station Number",
    "Latitude": "Destination Latitude",
    "Longitude": "Destination Longitude",
    "Constituency": "Destination Constituency"
}).drop(columns=["Station Name"])

mtr_data = mtr_data[[
    "Origin Station", "Origin Station Number", "Origin Latitude","Origin Longitude", "Origin Constituency","Origin District",
    "Destination Station", "Destination Station Number", "Destination Latitude","Destination Longitude","Destination Constituency","Destination District",
    "Total Travel Time (mins)",
    "Total Distance (km)",
    "Fare (HKD)",
    "No. of Stations",
    "Full Route"
]]

mtr_data.to_csv("MTR_Data/mtr_data.csv", index=False)

# Revise MTR_Data into unified format 

In [ ]:
def transform_data(df):
    # Transforms the input DataFrame into the universal format
    transformed_data = {
        "TRANSPORT": ["MTR"] * len(df),
        "ROUTE_ID": [0] * len(df),
        "COMPANY_CODE": ["MTR"] * len(df),
        "ROUTE_NAME": [0] * len(df),
        "CIRCULAR": [False] * len(df),
        "ROUTE_SEQ": [0] * len(df),
        "STOP_SEQ": [0] * len(df),
        "STOP_ID": df["Origin Station Number"].tolist(),
        "STOP_X": df["Origin Longitude"].tolist(),
        "STOP_Y": df["Origin Latitude"].tolist(),
        "STOP_PICK_DROP": [0] * len(df),
        "STOP_NAME": df["Origin Station"].tolist(),
        "DISTRICT": df["Origin Constituency"].tolist(),
        "NEXT_STOP_SEQ": [0] * len(df),
        "NEXT_STOP_ID": df["Destination Station Number"].tolist(),
        "NEXT_STOP_X": df["Destination Longitude"].tolist(),
        "NEXT_STOP_Y": df["Destination Latitude"].tolist(),
        "NEXT_STOP_NAME": df["Destination Station"].tolist(),
        "NEXT_DISTRICT": df["Destination Constituency"].tolist(),
        "DISTANCE": df["Total Distance (km)"].tolist(),
        "PRICE": df["Fare (HKD)"].tolist(),
    }
    
    return pd.DataFrame(transformed_data)

output_file_path = 'MTR_Data/revised_mtr_data.csv'
df = pd.read_csv('MTR_Data/mtr_data.csv')
transformed_df = transform_data(df)
transformed_df.to_csv(output_file_path, index=False)
print(f"Transformed data has been written to {output_file_path}")